# Kernel Methods and RKHS

| Difficulty | ███████░░░ 7/10 |
| :--- | :--- |
| Prerequisites | 11.01 |
| Time | ~80 min |
| Colab GPU | Not needed |

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/NiravRVaghasiya/maths-for-ai/blob/main/11_Functional_Analysis/02_kernel_methods_rkhs.ipynb)

---

## 🎯 Learning Objective

Learn Reproducing Kernel Hilbert Spaces, Mercer's theorem, and the representer theorem behind the kernel trick.

---

## 📐 Theory

`08.01` introduced the kernel trick as a Gram-matrix property: replace an inner product
$\langle\phi(\mathbf{x}),\phi(\mathbf{y})\rangle$ in some feature space $\phi$ with a cheap
function $k(\mathbf{x},\mathbf{y})$ that provably equals it. This notebook makes precise *which*
functions $k$ are actually allowed, what feature space they secretly correspond to, and why an
optimal model built from one always collapses to a finite, tractable form no matter how
infinite-dimensional that hidden feature space is.

### Reproducing Kernel Hilbert Spaces (RKHS)

Given a kernel $k$, its **Reproducing Kernel Hilbert Space** $\mathcal{H}_k$ is the (possibly
infinite-dimensional) Hilbert space of functions built from $k$, with the defining
**reproducing property**: $f(\mathbf{x}) = \langle f, k(\cdot,\mathbf{x})\rangle_{\mathcal{H}_k}$
for every $f\in\mathcal{H}_k$ — evaluating $f$ at a point is literally the same operation as
taking an inner product with the kernel centered there. The RKHS norm $\|f\|_{\mathcal{H}_k}$
measures a function's "complexity" with respect to $k$: functions that vary smoothly and
predictably (matching the kernel's own smoothness) have small norm, wildly oscillating functions
have large norm — this is exactly the quantity that regularization in kernel methods penalizes.

### Which functions are valid kernels: Moore–Aronszajn, not Mercer

Not every two-argument function is a valid kernel. The **Moore–Aronszajn theorem** is the
general result answering which ones are: $k$ corresponds to an actual inner product in some
feature space *if and only if* $k$ is symmetric and **positive semi-definite** — for every
finite set of points, the Gram matrix $K_{ij}=k(\mathbf{x}_i,\mathbf{x}_j)$ must be PSD
(recall PSD from `01.06`/`08.01`) — confirmed below, alongside a genuine non-example
($-\|\mathbf{x}-\mathbf{y}\|$, which fails this test) to show the condition actually bites.
This holds for *any* set $\mathcal{X}$ (no topology or compactness assumption needed) and is
exactly what guarantees an RKHS $\mathcal{H}_k$ exists at all, with an (implicit) feature map
$\phi$ satisfying $k(\mathbf{x},\mathbf{y})=\langle\phi(\mathbf{x}),\phi(\mathbf{y})\rangle$ —
often infinite-dimensional, as we'll confirm concretely for the Gaussian (RBF) kernel below,
whose feature map turns out to be an infinite power series.

**Mercer's theorem is a related but different, more specific result** — easy to conflate with
the one above since both concern PSD kernels and feature maps, but they are not the same
theorem. Mercer's theorem additionally requires $k$ to be *continuous* on a *compact* domain,
and in exchange gives something Moore–Aronszajn does not: an explicit, orthonormal
eigenfunction expansion $k(\mathbf{x},\mathbf{y}) = \sum_i \lambda_i\,\psi_i(\mathbf{x})
\psi_i(\mathbf{y})$ with $\lambda_i\ge 0$, convergent uniformly. The RBF kernel used below is
checked for PSD-ness on all of $\mathbb{R}^n$ — an unbounded, non-compact domain, so it is
Moore–Aronszajn (not Mercer's compactness-dependent statement) that actually licenses calling
it a valid kernel there.

### The representer theorem

Given data $(\mathbf{x}_i, y_i)_{i=1}^n$ and the regularized objective
$\min_{f\in\mathcal{H}_k} \sum_i \ell(y_i, f(\mathbf{x}_i)) + \lambda\|f\|_{\mathcal{H}_k}^2$ —
minimizing over the *entire*, possibly infinite-dimensional RKHS — the **representer theorem**
guarantees the minimizer always has the finite form $f^\star(\mathbf{x}) = \sum_{i=1}^n
\alpha_i\, k(\mathbf{x},\mathbf{x}_i)$: a combination of just $n$ kernel functions, one per
training point, regardless of how large (even infinite) $\mathcal{H}_k$ is. This is the precise
reason kernel methods are tractable at all — optimizing over an infinite-dimensional function
space reduces to optimizing $n$ real numbers $\alpha_i$, verified below by confirming this finite
form actually achieves the true minimum (matching direct numerical optimization over the
$\alpha_i$).

### Why this matters beyond kernel SVMs

The representer theorem plus Moore–Aronszajn together explain *why* the kernel trick from
`08.01` works algorithmically, not just algebraically: any kernel-based learning problem reduces
to $n\times n$ linear algebra on the Gram matrix, no matter how rich the implicit feature space
$\phi$ is. `08.01` already showed attention's $QK^\top$ is a Gram matrix; this notebook's closing
section pushes that connection further, toward the actual algorithmic payoff — linear-time
attention via explicit, finite kernel feature maps.

---

In [ ]:
# Setup
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline
plt.style.use('seaborn-v0_8-whitegrid')

## 👁️ Visual Intuition

Fitting the same noisy data with a kernel ridge regression at increasing regularization strength
$\lambda$ shows the RKHS norm penalty doing exactly what it claims: larger $\lambda$ forces a
smaller $\|f\|_{\mathcal{H}_k}^2$, which visibly flattens and smooths the fitted function.

In [ ]:
rng_viz = np.random.default_rng(0)

def rbf_kernel_1d(x1, x2, gamma=2.0):
    return np.exp(-gamma * (x1[:, None] - x2[None, :])**2)

n_viz = 12
X_train_viz = np.sort(rng_viz.uniform(-3, 3, n_viz))
y_train_viz = np.sin(X_train_viz) + rng_viz.normal(scale=0.15, size=n_viz)
x_grid_viz = np.linspace(-3.5, 3.5, 300)

def fit_and_predict(lam):
    K = rbf_kernel_1d(X_train_viz, X_train_viz)
    alpha = np.linalg.solve(K + lam * np.eye(n_viz), y_train_viz)
    K_grid = rbf_kernel_1d(x_grid_viz, X_train_viz)
    f_grid = K_grid @ alpha
    rkhs_norm_sq = alpha @ K @ alpha
    return f_grid, rkhs_norm_sq

fig, ax = plt.subplots(figsize=(8.5, 5.5))
ax.scatter(X_train_viz, y_train_viz, color='k', zorder=5, s=40, label='training data')
ax.plot(x_grid_viz, np.sin(x_grid_viz), color='gray', ls=':', lw=1.5, label='true $\\sin(x)$')

lambdas = [0.001, 0.05, 1.0]
colors = ["#C44E52", "#DD8452", "#4C72B0"]
for lam, color in zip(lambdas, colors):
    f_grid, norm_sq = fit_and_predict(lam)
    ax.plot(x_grid_viz, f_grid, color=color, lw=2, label=f"$\\lambda$={lam}, $\\|f\\|_H^2$={norm_sq:.2f}")

ax.set_ylim(-2.5, 2.0)
ax.set_title("Larger regularization $\\lambda$ forces a smaller RKHS norm,\nvisibly smoothing/flattening the fitted function")
ax.legend(fontsize=9)
plt.show()

for lam in lambdas:
    _, norm_sq = fit_and_predict(lam)
    print(f"lambda={lam}: RKHS norm^2 = {norm_sq:.4f}")
print("\nRKHS norm shrinks monotonically as lambda grows -- the regularizer is doing exactly")
print("what its name promises: penalizing function 'complexity' as measured by this specific")
print("kernel-induced notion of norm, not some vague informal sense of wiggliness.")

## 🐍 Implementation from Scratch

We verify the Moore–Aronszajn PSD condition distinguishes valid kernels from invalid ones, confirm the RBF
kernel's implicit feature map is genuinely infinite-dimensional (a truncated approximation only
converges as more terms are added, never matches exactly), and verify the representer theorem's
claim that the true minimizer over an entire (infinite) RKHS coincides with the finite $n$-term
kernel expansion.

In [ ]:
import math
from scipy.optimize import minimize

rng = np.random.default_rng(0)

# --- Moore–Aronszajn: PSD-ness distinguishes valid kernels from invalid ones ---
def rbf_kernel(X, Y, gamma=1.0):
    sq_dists = np.sum(X**2, axis=1)[:, None] + np.sum(Y**2, axis=1)[None, :] - 2 * X @ Y.T
    return np.exp(-gamma * np.maximum(sq_dists, 0))

def negative_distance_kernel(X, Y):
    # k(x,y) = -||x-y|| -- looks like a reasonable "similarity" function, but is NOT PSD in general
    sq_dists = np.sum(X**2, axis=1)[:, None] + np.sum(Y**2, axis=1)[None, :] - 2 * X @ Y.T
    return -np.sqrt(np.maximum(sq_dists, 0))

X_mercer = rng.normal(size=(30, 2))
eig_rbf = np.linalg.eigvalsh(rbf_kernel(X_mercer, X_mercer))
eig_bad = np.linalg.eigvalsh(negative_distance_kernel(X_mercer, X_mercer))

print("Moore–Aronszajn PSD check (Gram matrix must be PSD for a valid kernel):")
print(f"  RBF kernel: min eigenvalue = {eig_rbf.min():.6f}  (PSD: {eig_rbf.min() >= -1e-8})")
print(f"  '-||x-y||' kernel: min eigenvalue = {eig_bad.min():.4f}  (PSD: {eig_bad.min() >= -1e-8})")
print(f"  The negative-distance function LOOKS like a similarity measure but fails the")
print(f"  Moore–Aronszajn PSD condition -- it does NOT correspond to any actual inner")
print(f"  product in any feature space.")

# --- The RBF kernel's implicit feature map is genuinely infinite-dimensional ---
def rbf_kernel_scalar(x, y, gamma=1.0):
    return np.exp(-gamma * (x - y)**2)

def rbf_taylor_feature_map(x, gamma, n_terms):
    # exp(-gamma(x-y)^2) = exp(-gamma x^2) exp(-gamma y^2) exp(2 gamma x y), and
    # exp(2 gamma x y) = sum_k (2 gamma x y)^k / k!  -- each term of this series is one feature
    k = np.arange(n_terms)
    coeffs = (2 * gamma) ** k / np.array([math.factorial(kk) for kk in k])
    return np.exp(-gamma * x**2) * np.sqrt(coeffs) * x**k

x0, y0, gamma = 0.6, -0.3, 1.0
true_val = rbf_kernel_scalar(x0, y0, gamma)
print(f"\nRBF kernel value exp(-gamma(x-y)^2) at x={x0}, y={y0}: {true_val:.8f}")
for n_terms in [1, 3, 5, 10, 20]:
    phi_x = rbf_taylor_feature_map(np.array(x0), gamma, n_terms)
    phi_y = rbf_taylor_feature_map(np.array(y0), gamma, n_terms)
    approx = np.dot(phi_x, phi_y)
    print(f"  truncated to {n_terms:2d} terms: feature-map dot product = {approx:.8f}")
print("The truncated feature map only CONVERGES to the true kernel value as more terms are")
print("added -- the RBF kernel's actual feature map has infinitely many components.")

# --- The representer theorem: the true RKHS minimizer coincides with the n-term kernel expansion ---
n = 15
X_rep = rng.normal(size=(n, 2))
y_rep = np.sin(X_rep[:, 0] * 2) + 0.3 * X_rep[:, 1] + rng.normal(scale=0.1, size=n)
K = rbf_kernel(X_rep, X_rep, gamma=0.5)
lam = 0.1

alpha_closed_form = np.linalg.solve(K + lam * np.eye(n), y_rep)

def objective(alpha):
    f_train = K @ alpha
    return np.sum((y_rep - f_train)**2) + lam * (alpha @ K @ alpha)

result = minimize(objective, x0=rng.normal(size=n), method='BFGS', options={'maxiter': 5000, 'gtol': 1e-10})
alpha_numeric = result.x

print(f"\nRepresenter theorem check -- minimizing sum(loss) + lambda*||f||_H^2:")
print(f"  Objective at closed-form n-term kernel expansion: {objective(alpha_closed_form):.8f}")
print(f"  Objective at numerically optimized alpha:         {objective(alpha_numeric):.8f}")
print(f"  Same minimum reached: {np.isclose(objective(alpha_closed_form), objective(alpha_numeric), atol=1e-4)}")
print(f"  (K has a near-zero eigenvalue, so alpha itself isn't unique -- but the achieved")
print(f"  minimum objective value is, confirming the finite n-term form really does minimize")
print(f"  over the ENTIRE possibly-infinite-dimensional RKHS, not just some restricted subset.)")

## 🤖 Why This Matters for AI

Kernel methods have a direct, practical descendant in modern efficient-transformer research.
`08.01` noted attention's $QK^\top$ is a Gram matrix; softmax attention specifically corresponds
to a kernel $k(\mathbf{q},\mathbf{k}) \propto \exp(\mathbf{q}\cdot\mathbf{k}/\sqrt{d_k})$, which
happens to be one more member of the exponential-kernel family this notebook studies via the
RBF kernel above — and just like the RBF kernel, its implicit feature map is infinite-dimensional,
which is exactly why standard attention has to compute the full $n\times n$ score matrix
explicitly rather than factoring it. **Linear attention** (e.g. Performers' FAVOR+) applies this
notebook's core idea directly: approximate that same exponential kernel with an *explicit*,
finite random feature map $\phi$, so that $\text{softmax}(QK^\top)V \approx \phi(Q)\big(\phi(K)^\top
V\big)$ — reassociating the computation to avoid ever forming the $n\times n$ matrix, dropping
the cost from quadratic to linear in sequence length. Below, we implement this random feature
map from scratch, confirm it approximates the true exponential kernel value (converging as more
random features are used, exactly like the RBF Taylor expansion above), and confirm the
resulting linear-attention output converges to standard softmax attention's output.

In [ ]:
rng_attn = np.random.default_rng(0)

def softmax_rows(Z):
    Z = Z - Z.max(axis=1, keepdims=True)
    e = np.exp(Z)
    return e / e.sum(axis=1, keepdims=True)

def standard_attention(Q, K, V):
    d_k = Q.shape[-1]
    scores = Q @ K.T / np.sqrt(d_k)
    weights = softmax_rows(scores)
    return weights @ V

def random_features_map(X, n_features, d_k, seed):
    # Positive random features (Performers-style) approximating exp(x.y), built the same way
    # as this notebook's RBF Taylor-series feature map -- an explicit, FINITE stand-in for an
    # infinite-dimensional kernel feature space.
    r = np.random.default_rng(seed)
    omega = r.normal(size=(d_k, n_features))
    X_scaled = X / (d_k ** 0.25)
    proj = X_scaled @ omega
    norm_sq = np.sum(X_scaled**2, axis=1, keepdims=True)
    return np.exp(proj - norm_sq / 2) / np.sqrt(n_features)

def linear_attention(Q, K, V, n_features, seed):
    d_k = Q.shape[-1]
    phi_Q = random_features_map(Q, n_features, d_k, seed)
    phi_K = random_features_map(K, n_features, d_k, seed)  # SAME random projection for Q and K
    # Reassociated computation: phi_Q @ (phi_K^T @ V) never forms the seq_len x seq_len matrix
    numerator = phi_Q @ (phi_K.T @ V)
    denominator = phi_Q @ phi_K.sum(axis=0)
    return numerator / denominator[:, None]

# First confirm the KERNEL approximation itself (not yet full attention): does phi(q).phi(k)
# approximate exp(q.k/sqrt(d_k)) for individual vectors, with realistic (post-LayerNorm-scale) inputs?
d_k = 8
q_single = rng_attn.normal(scale=0.3, size=(1, d_k))
k_single = rng_attn.normal(scale=0.3, size=(1, d_k))
true_kernel_val = np.exp((q_single @ k_single.T)[0, 0] / np.sqrt(d_k))
print(f"True kernel value exp(q.k/sqrt(d_k)) for one (q,k) pair: {true_kernel_val:.4f}")
for n_feat in [16, 256, 4096]:
    phi_q = random_features_map(q_single, n_feat, d_k, seed=1)
    phi_k = random_features_map(k_single, n_feat, d_k, seed=1)
    approx = (phi_q @ phi_k.T)[0, 0]
    print(f"  n_features={n_feat:5d}: random-feature approximation = {approx:.4f}")

print("\n--- Full attention output: standard softmax vs. linear (kernelized) attention ---")
seq_len = 12
Q = rng_attn.normal(scale=0.3, size=(seq_len, d_k))
K = rng_attn.normal(scale=0.3, size=(seq_len, d_k))
V = rng_attn.normal(size=(seq_len, 6))
out_standard = standard_attention(Q, K, V)

feature_counts = [16, 64, 256, 1024, 4096]
mean_errors = []
for n_features in feature_counts:
    errs = [np.linalg.norm(linear_attention(Q, K, V, n_features, seed=s) - out_standard) / np.linalg.norm(out_standard)
            for s in range(5)]
    mean_errors.append(np.mean(errs))
    print(f"n_random_features={n_features:5d}: mean relative error over 5 random seeds = {np.mean(errs):.4f}")

fig, ax = plt.subplots(figsize=(7.5, 5))
ax.loglog(feature_counts, mean_errors, 'o-', color="#4C72B0", lw=2, ms=8)
ax.set_xlabel("number of random features"); ax.set_ylabel("relative error vs. standard softmax attention")
ax.set_title("Linear attention's error shrinks as more random\nfeatures approximate the exponential kernel more precisely")
plt.show()

seq_len_real, d_k_real, n_feat_real = 4096, 128, 256
standard_cost = seq_len_real ** 2 * d_k_real
linear_cost = seq_len_real * n_feat_real * d_k_real
print(f"\nAt a realistic scale (seq_len={seq_len_real}, d_k={d_k_real}, {n_feat_real} random features):")
print(f"  Standard attention cost ~ seq_len^2 * d_k = {standard_cost:,}")
print(f"  Linear attention cost ~ seq_len * n_features * d_k = {linear_cost:,}")
print(f"  Reduction: {standard_cost/linear_cost:.0f}x fewer operations, at the price of the")
print(f"  approximation error measured above -- the SAME finite-vs-infinite tradeoff this")
print(f"  notebook's RBF Taylor-series example made explicit.")

## 🏋️ Exercises

Six levels, in order: mechanical computation → conceptual understanding → derivation →
implementation → application → open-ended challenge. Solutions/outlines are in the collapsed
`Solution` blocks below each exercise — try before revealing.

### 🔢 COMPUTE
1. Using $k(\mathbf{x},\mathbf{y})=e^{-\gamma\|\mathbf{x}-\mathbf{y}\|^2}$ with $\gamma=1$,
   compute by hand the $3\times3$ Gram matrix for $\mathbf{x}_1=(0,0)$, $\mathbf{x}_2=(1,0)$,
   $\mathbf{x}_3=(0,1)$, then confirm with `np.linalg.eigvalsh` that every eigenvalue is
   non-negative — the Moore–Aronszajn PSD sanity check. Then build the candidate function
   $k'(\mathbf{x},\mathbf{y})=\mathbf{x}^\top\mathbf{y}+1$ if $x_1>y_1$ else $0$ on the same
   three points, and confirm its matrix isn't even symmetric — the more basic of the two
   Moore–Aronszajn requirements ("symmetric *and* PSD") to fail.

<details>
<summary>💡 Solution</summary>

Pairwise squared distances are $\|\mathbf{x}_1-\mathbf{x}_2\|^2=\|\mathbf{x}_1-\mathbf{x}_3\|^2=1$
and $\|\mathbf{x}_2-\mathbf{x}_3\|^2=2$, so
$$K=\begin{pmatrix}1 & e^{-1} & e^{-1}\\ e^{-1} & 1 & e^{-2}\\ e^{-1} & e^{-2} & 1\end{pmatrix}
\approx\begin{pmatrix}1 & 0.3679 & 0.3679\\ 0.3679 & 1 & 0.1353\\ 0.3679 & 0.1353 & 1\end{pmatrix}.$$
Its eigenvalues are approximately $0.543$, $0.865$, $1.592$ — all positive, confirming PSD (in
fact positive*definite* here, a strictly stronger property PSD only requires as $\ge 0$).
For $k'$: only the pair $(\mathbf{x}_2,\mathbf{x}_3)$ has $x_1$-coordinate of the first argument
($1$) exceeding that of the second ($0$), and symmetrically $(\mathbf{x}_2,\mathbf{x}_1)$, giving
$$K'=\begin{pmatrix}0&0&0\\1&0&1\\0&0&0\end{pmatrix},$$
which is visibly not symmetric ($K'_{12}=0\ne K'_{21}=1$) — note this is the Moore–Aronszajn
condition specifically (symmetric + PSD on every finite Gram matrix), the theorem this notebook's
Theory section is careful to distinguish from the separate, compactness-dependent Mercer's
theorem discussed in UNDERSTAND below.

</details>

### 💭 UNDERSTAND
2. The notebook checks the RBF kernel's Moore–Aronszajn PSD condition on `X_mercer`, 30 points
   drawn from all of $\mathbb{R}^2$ via `rng.normal(...)`, and explicitly attributes the RBF
   kernel's validity there to Moore–Aronszajn rather than to Mercer's theorem. Explain precisely
   why Mercer's theorem's own hypotheses could not have been invoked to reach the same
   conclusion on this domain — even though the actual check performed (a finite Gram matrix's
   eigenvalues) looks identical to a step in a Mercer-style argument.

<details>
<summary>💡 Solution</summary>

Mercer's theorem requires the kernel to be continuous on a *compact* domain, and in exchange
delivers something stronger: an explicit orthonormal eigenfunction expansion
$k(\mathbf{x},\mathbf{y})=\sum_i\lambda_i\psi_i(\mathbf{x})\psi_i(\mathbf{y})$. $\mathbb{R}^2$ is
closed but unbounded, so by Heine–Borel it is not compact — Mercer's hypothesis fails outright,
independent of anything about the kernel itself or how many points get sampled. Moore–Aronszajn
makes no topological assumption on the domain at all (it holds "for any set $\mathcal{X}$"): a
kernel is valid there purely because it's symmetric and PSD on every finite Gram matrix, which is
both necessary and sufficient for an RKHS to exist. The eigenvalue check performed in the cell is
genuine finite-sample evidence for *that* (weaker-hypothesis) condition; it says nothing about
compactness one way or the other, since compactness is a structural fact about the domain
$\mathbb{R}^2$ itself, not something any numerical Gram-matrix computation could establish or
refute.

</details>

### ✏️ DERIVE
3. Derive the bound $|f(\mathbf{x})|\le\|f\|_{\mathcal{H}_k}\sqrt{k(\mathbf{x},\mathbf{x})}$ that
   follows directly from the reproducing property, then check it numerically against the fitted
   function from the RKHS-norm visualization at $\lambda=0.05$.

<details>
<summary>💡 Solution outline</summary>

Start from the reproducing property applied to $f$ itself: $f(\mathbf{x}) = \langle f,
k(\cdot,\mathbf{x})\rangle_{\mathcal{H}_k}$. Apply Cauchy–Schwarz to the right-hand side:
$$|f(\mathbf{x})| = \big|\langle f, k(\cdot,\mathbf{x})\rangle_{\mathcal{H}_k}\big| \le
\|f\|_{\mathcal{H}_k}\,\|k(\cdot,\mathbf{x})\|_{\mathcal{H}_k}.$$
Now apply the reproducing property a *second* time, to the function $k(\cdot,\mathbf{x})$
itself, evaluated at $\mathbf{x}$: $k(\mathbf{x},\mathbf{x}) = \langle k(\cdot,\mathbf{x}),
k(\cdot,\mathbf{x})\rangle_{\mathcal{H}_k} = \|k(\cdot,\mathbf{x})\|_{\mathcal{H}_k}^2$, i.e.
$\|k(\cdot,\mathbf{x})\|_{\mathcal{H}_k}=\sqrt{k(\mathbf{x},\mathbf{x})}$. Substituting gives
$$|f(\mathbf{x})| \le \|f\|_{\mathcal{H}_k}\sqrt{k(\mathbf{x},\mathbf{x})}.$$
Numeric check: for `fit_and_predict(0.05)`, $\|f\|_{\mathcal{H}_k}=\sqrt{\alpha^\top K\alpha}
\approx 1.72$. The RBF kernel always has $k(x,x)=e^0=1$, so the bound simplifies to
$|f(x)|\lesssim1.72$ everywhere on $\mathbb{R}$. Evaluating the fitted $f$ at several points
(training and non-training alike, e.g. $x=0, 1.5, -2, 3.4$) gives magnitudes no larger than about
$1.14$ — comfortably inside the bound, as the derivation guarantees for *every* $x$, not just
the ones checked.

</details>

### 🐍 IMPLEMENT
4. The notebook's `rbf_taylor_feature_map` only handles a scalar input. Extend it to
   $2$-dimensional inputs by building the feature map for `rbf_kernel(x, y, gamma) =
   exp(-gamma*||x-y||^2)` as a Kronecker product of two independent copies of the 1D feature map
   (one per coordinate), and check that the dot product of the resulting 2D feature vectors
   converges to the true 2D RBF kernel value as the number of per-dimension terms grows.

<details>
<summary>✅ How to know you're right</summary>

The 2D RBF kernel factors exactly as a product of two 1D RBF kernels,
$e^{-\gamma((x_1-y_1)^2+(x_2-y_2)^2)} = e^{-\gamma(x_1-y_1)^2}\cdot e^{-\gamma(x_2-y_2)^2}$, and a
Kronecker product of feature maps reproduces exactly this factorization:
$\langle\phi(x_1)\otimes\phi(x_2),\,\phi(y_1)\otimes\phi(y_2)\rangle = \langle\phi(x_1),
\phi(y_1)\rangle\cdot\langle\phi(x_2),\phi(y_2)\rangle$. Checked against
`np.exp(-gamma * np.sum((x-y)**2))` directly, the approximation should converge geometrically
with the number of per-dimension terms — by around $8$-$12$ terms per dimension (a feature
vector of dimension $64$-$144$), the error should already be at or near float64 machine
precision (order $10^{-9}$ to $10^{-15}$), the same convergence behavior the notebook's own 1D
version shows, just now on a genuinely 2-dimensional input.

</details>

### 🤖 APPLY
5. Kernel ridge regression (`fit_and_predict`) and Gaussian Process regression are usually taught
   as separate frameworks — one a regularized-optimization/RKHS story, the other a probabilistic
   inference story. Fit an `sklearn.gaussian_process.GaussianProcessRegressor` with a matching
   RBF kernel (convert `gamma` to sklearn's `length_scale` convention,
   $k(x,y)=\exp(-d(x,y)^2/(2\ell^2))$, so $\ell=\sqrt{1/(2\gamma)}$), `alpha` set to the same
   $\lambda$, and its internal optimizer disabled, on the same training data, and compare its
   posterior mean predictions on the grid to `fit_and_predict`'s output.

<details>
<summary>✅ What you should observe</summary>

The two predictions should agree to essentially machine precision (relative difference on the
order of $10^{-15}$-$10^{-16}$, not just "close"). This is because kernel ridge regression's
closed-form solution and the GP posterior mean, under a zero-mean GP prior with kernel $k$ and
i.i.d. Gaussian observation noise of variance $\lambda$, are two different derivations
(regularized-RKHS-optimization vs. Bayesian-conditioning) of the literal same estimator. This
notebook's `fit_and_predict` has silently been computing a Gaussian process posterior mean the
entire time, whether or not probability was ever mentioned.

</details>

### 🚀 CHALLENGE
6. The AI section's linear-attention approximation error stays in the single-digit-percent range
   even at hundreds to thousands of random features. Investigate how this relative error behaves
   as `seq_len` grows from `12` up to several hundred, holding `n_features` fixed at a couple of
   values. Does the error systematically increase, decrease, or stay roughly flat? Relate your
   finding to whether the per-$(q,k)$-pair kernel approximation quality (already checked in
   isolation, for a single pair, earlier in the AI section) should itself depend on sequence
   length, versus what changes when errors are aggregated across an entire $n\times d_v$
   attention output via a relative Frobenius-type norm.

<details>
<summary>🔍 What a strong answer covers</summary>

A strong answer sweeps `seq_len` across roughly two orders of magnitude (e.g. `12` to `384`) at
fixed `n_features` and reports that the relative error shows no clear systematic trend with
`seq_len` — it fluctuates in a roughly stable band (empirically, mean relative error stays
roughly in the 3%-5% range at `n_features=256` across that whole range) rather than growing or
shrinking monotonically. The explanation: the per-$(q,k)$-pair random-feature approximation
error is a property of the kernel step alone, fixed by `n_features`, $d_k$, and the input scale —
it doesn't know how many other tokens are in the sequence. Aggregating many roughly-independent
per-pair errors over a longer sequence changes the statistics of the *relative-error estimate*
itself (both the numerator and denominator norms are sums over more terms) but doesn't introduce
a new source of bias tied to sequence length. A strong answer connects this to why it matters for
Performers-style linear attention specifically: the approximation quality per random-feature
budget doesn't get harder to control as the model handles longer contexts, which is exactly what
makes trading quadratic cost for linear cost worthwhile rather than a hidden long-context penalty.

</details>

---

## 📚 Further Reading
- Schölkopf & Smola, *Learning with Kernels*, Ch. 2-4
- Berlinet & Thomas-Agnan, *Reproducing Kernel Hilbert Spaces in Probability and Statistics*
- Choromanski et al., [\"Rethinking Attention with Performers\"](https://arxiv.org/abs/2009.14794)

---

*Next notebook: [The Neural Tangent Kernel](03_neural_tangent_kernel.ipynb)*